In [2]:
import pandas as pd
import numpy as np

In [3]:
trips = pd.read_csv("./data/trips_small.csv", sep = ',')
cities = pd.read_csv("./data/cities.csv", sep = ',')
bike_types = pd.read_csv("./data/bike_types.csv", sep = ',')
bikes = pd.read_csv("./data/bikes.csv", sep = ',')
stations = pd.read_csv("./data/stations.csv", sep = ',')
station_status = pd.read_csv("./data/stations_status.csv", sep = ',')

In [4]:
datasets = {
    "trips": trips,
    "cities": cities,
    "bike_types": bike_types,
    "bikes": bikes,
    "stations": stations,
    "station_status": station_status,
}

def describe_df(name, df, n=5):
    print(f"\n{'=' * 80}\n{name}  —  shape: {df.shape}\n{'=' * 80}")

    print("\n-- head --")
    print(df.head(n).to_string())

    print("\n-- column overview --")
    overview = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "na_count": df.isna().sum(),
        "na_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })
    print(overview.to_string())

    print("\n-- summary (numeric) --")
    num = df.select_dtypes(include="number")
    print(num.describe().T.to_string() if not num.empty else "(no numeric columns)")

    print("\n-- summary (non-numeric) --")
    obj = df.select_dtypes(exclude="number")
    print(obj.describe().T.to_string() if not obj.empty else "(no non-numeric columns)")

for name, df in datasets.items():
    describe_df(name, df)


trips  —  shape: (1161376, 13)

-- head --
   bike_id  city_id    time_start  lon_start  lat_start   lon_end    lat_end  station_id_start  station_id_end  battery_start  battery_end  duration     distance
0    11462        7  1.669377e+09   8.253932  50.069332  8.253393  50.070504       126084480.0             NaN            NaN          NaN     360.0   135.879806
1    11462        7  1.669897e+09   8.242628  50.071337  8.216180  50.095700        87796472.0       4775153.0            NaN          NaN    1920.0  3301.514764
2    11462        7  1.670252e+09   8.216180  50.095700  8.217791  50.095744         4775153.0             NaN            NaN          NaN     780.0   115.020620
3    11462        7  1.670349e+09   8.226448  50.076546  8.249343  50.078160         4775087.0      87796883.0            NaN          NaN     540.0  1643.609250
4    11462        7  1.671514e+09   8.249343  50.078160  8.242628  50.071337        87796883.0      87796472.0            NaN          NaN    4680

In [5]:
bike_types.dropna().count()

id                  10
vehicle_image       10
name                10
description         10
form_factor         10
rider_capacity      10
propulsion_type     10
max_range           10
battery_capacity    10
dtype: int64

In [6]:
bike_types.dropna(subset=["propulsion_type"])

,id,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity
24,174,https://static.nextbike.net/app/biketypes/type...,Comfort,NaN,bicycle,1.0,human,NaN,NaN
31,111,https://static.nextbike.net/app/biketypes/type...,Rower z fotelikiem dziecięcym,NaN,bicycle,2.0,human,NaN,NaN
32,9,https://static.nextbike.net/app/biketypes/type...,Tandem,- MountainBike Rahmen ALU\r\n- Shimano 27-Gang...,bicycle,2.0,human,NaN,NaN
33,170,https://static.nextbike.net/app/biketypes/type...,Rower z fotelikiem dziecięcym,NaN,bicycle,2.0,human,NaN,NaN
34,119,https://static.nextbike.net/app/biketypes/type...,Elektricno kolo,NaN,bicycle,1.0,electric_assist,60000.0,500.0
...,...,...,...,...,...,...,...,...,...
116,199,https://static.nextbike.net/app/biketypes/type...,3-rychlostní SMARTbike,3-rychlostní kola bez podpory města,bicycle,1.0,human,NaN,NaN
117,200,https://static.nextbike.net/app/biketypes/type...,8-rychlostní SMARTbike,Nová osmirychlostní kola bez podpory města,bicycle,1.0,human,NaN,NaN
118,196,https://static.nextbike.net/app/biketypes/type...,with Framelock,SMARTbike 2.0 with labeling\r\n\r\n- eco frame...,bicycle,1.0,human,NaN,NaN
119,150,https://static.nextbike.net/app/biketypes/type...,with Framelock,NaN,bicycle,1.0,human,NaN,NaN


In [7]:
bike_types.query("propulsion_type == 'electric_assist'").isna().sum()

id                   0
vehicle_image        0
name                 0
description          5
form_factor          0
rider_capacity       0
propulsion_type      0
max_range           12
battery_capacity     5
dtype: int64

In [8]:
len(bike_types['vehicle_image'].unique())

92

In [33]:
#bike_types.query("battery_capacity.notnull() & propulsion_type.isnull()")
#bike_types.query("max_range.notnull() & propulsion_type.isnull()")
bike_types.query("battery_capacity.isnull() & propulsion_type == 'electric_assist'")
bike_types.query("max_range.isnull() & propulsion_type == 'electric_assist'")

,id,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity
37,179,https://static.nextbike.net/app/biketypes/type...,e-CARGObike,"- PEDALPOWER ""eHarry"" pl0072 Cargo\r\n - BROS...",cargo_bicycle,1.0,electric_assist,NaN,NaN
63,121,https://static.nextbike.net/app/biketypes/type...,e-CARGObike,"- PEDALPOWER ""eHarry"" pl0072 Cargo\r\n - BROS...",cargo_bicycle,1.0,electric_assist,NaN,630.0
66,232,https://static.nextbike.net/app/biketypes/type...,e-Cargobike,"DOLLY BIKES ""e-Dolly"" MaxDrive\r\n\r\n - BAFAN...",cargo_bicycle,1.0,electric_assist,NaN,600.0
67,233,https://static.nextbike.net/app/biketypes/type...,e-Cargobike,"DOLLY BIKES ""e-Dolly"" MaxDrive\r\n\r\n - BAFAN...",cargo_bicycle,1.0,electric_assist,NaN,600.0
68,226,https://static.nextbike.net/app/biketypes/type...,e-CARGObike,- E-Dolly Family Mid-engine 2-Wheel bakfiets b...,cargo_bicycle,1.0,electric_assist,NaN,600.0
73,224,https://static.nextbike.net/app/biketypes/type...,e-CARGObike,"- PEDALPOWER ""eHarry"" pl0072 Cargo\r\n - BROS...",cargo_bicycle,1.0,electric_assist,NaN,630.0
85,181,https://static.nextbike.net/app/biketypes/type...,e-CARGObike,"- PEDALPOWER ""eHarry"" pl0072 Cargo\r\n - BROS...",cargo_bicycle,1.0,electric_assist,NaN,NaN
92,238,https://static.nextbike.net/app/biketypes/type...,e-Cargobike,"DOLLY BIKES ""e-Dolly"" MaxDrive\r\n\r\n - BAFAN...",cargo_bicycle,1.0,electric_assist,NaN,NaN
93,239,https://static.nextbike.net/app/biketypes/type...,e-Cargobike,"DOLLY BIKES ""e-Dolly"" MaxDrive\r\n\r\n - BAFAN...",cargo_bicycle,1.0,electric_assist,NaN,600.0
94,240,https://static.nextbike.net/app/biketypes/type...,e-Cargobike,"DOLLY BIKES ""e-Dolly"" MaxDrive\r\n\r\n - BAFAN...",cargo_bicycle,1.0,electric_assist,NaN,NaN


In [22]:
bike_types['name'].dropna().unique()

<StringArray>
[                        'Comfort',   'Rower z fotelikiem dziecięcym',
                          'Tandem',                 'Elektricno kolo',
                     'e-CARGObike',            'Rowerek dziecięcy 6+',
     '3-rychlostní SMARTbike 2.0.',       'e-SMARTbike 2.0 with RFID',
         '28 Zoll - Tourenfahrrad',                'e-SMARTbike 2.0.',
          'Smartbike 2.0 (B) 2019',                 'e-SMARTbike 1.0',
                       'UsedomRad',               'UsedomRad Classic',
         '20 Zoll - Kinderfahrrad',         '24 Zoll - Kinderfahrrad',
                 'Z wałem kardana',          '7-rychlostní SMARTbike',
                 'e-Bike "LaBigi"',                          'E-Bike',
     '7-rychlostní SMARTbike 2.0.',                     'e-Cargobike',
                       'CARGObike',     '8-rychlostní SMARTbike 2.0.',
                  'with Framelock',          '8-rychlostní SMARTbike',
            'CARGObike (3 wheels)',               '+ Children's

In [34]:
cargo_bikes = bike_types.query("name in ['e-CARGObike', 'e-Cargobike']")
cargo_bikes.count()
cargo_bikes['vehicle_image'].unique()

<StringArray>
['https://static.nextbike.net/app/biketypes/type/vn/179/h500.png',
 'https://static.nextbike.net/app/biketypes/type/le/121/h500.png',
 'https://static.nextbike.net/app/biketypes/type/dg/232/h500.png',
 'https://static.nextbike.net/app/biketypes/type/dg/233/h500.png',
 'https://static.nextbike.net/app/biketypes/type/lt/226/h500.png',
 'https://static.nextbike.net/app/biketypes/type/ka/224/h500.png',
 'https://static.nextbike.net/app/biketypes/type/df/181/h500.png',
 'https://static.nextbike.net/app/biketypes/type/bf/238/h500.png',
 'https://static.nextbike.net/app/biketypes/type/bf/239/h500.png',
 'https://static.nextbike.net/app/biketypes/type/bf/240/h500.png',
 'https://static.nextbike.net/app/biketypes/type/bf/241/h500.png',
 'https://static.nextbike.net/app/biketypes/type/bf/242/h500.png']
Length: 12, dtype: str

In [41]:
bike_types['description'].count() # Length: 51
len(bike_types['description'].unique()) # 41 maybe these are worth combining?

41

In [51]:
# Okay lets try to find the bikes that have battery stats in trips so that we know for sure they are electrical
bikes_with_battery_stats = trips.query("battery_start.notnull() | battery_end.notnull()")['bike_id'].unique()
bike_type_ids_for_this = bikes.query("id in @bikes_with_battery_stats")['bike_type_id'].unique()
bike_types.query("propulsion_type.isnull() & id in @bike_type_ids_for_this")
# Okay so there are no bikes that have propulsion type as null 
# and have battery stats in trips so maybe they are all human powered

,id,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity


In [53]:
bike_types.query("propulsion_type.isnull()")

,id,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity
0,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,214,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,225,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
bike_types.dropna(subset=['form_factor']).groupby('form_factor').agg(['count'])

,id,vehicle_image,name,description,rider_capacity,propulsion_type,max_range,battery_capacity
,count,count,count,count,count,count,count,count
form_factor,,,,,,,,
bicycle,71,71,71,38,71,71,15,14
cargo_bicycle,19,19,19,13,19,19,0,8
other,1,1,1,0,1,1,0,0


In [81]:
# Lets check how many trips are there with the bikes with missing data, maybe it is safe to
# remove the ones that belong in cities that have a lot of bikes but we should be careful to not remove
# the ones that are our only samples in small cities
bike_types_with_na = bike_types.query("propulsion_type.isnull()")['id']
bikes_with_na = bikes.query("bike_type_id in @bike_types_with_na")['id']
bikes_with_na
trips.query("bike_id in @bikes_with_na").groupby('city_id').agg({'city_id' : ['count']})
# Okay there are significant number of cities so we keep them for now but maybe we can mark the data as unknown

,city_id
,count
city_id,
155,32
251,1
320,5
394,623
411,23
413,22
429,26
431,250


In [87]:
bikes.isna().sum()

id              0
bike_type_id    0
computer_id     0
dtype: int64

In [132]:
# are there any bike types here that do not exist in the bike type table?
bike_type_ids = bike_types['id']
bikes.query("bike_type_id not in @bike_type_ids")

,id,bike_type_id,computer_id,propulsion_type


In [97]:
# How many bikes of each type
bikes = bikes.merge(
    bike_types[['id', 'propulsion_type']].rename(columns={'id': 'bike_type_id'}), 
    on='bike_type_id')
bikes.groupby('propulsion_type').agg(['count'])

,id,bike_type_id,computer_id
,count,count,count
propulsion_type,,,
electric_assist,5856,5856,5856
human,81530,81530,81530


In [99]:
bikes.query("computer_id == 0 & propulsion_type == 'electric_assist'")

,id,bike_type_id,computer_id,propulsion_type
230,373270,204,0,electric_assist
1054,80234,85,0,electric_assist
1055,80208,85,0,electric_assist
1101,80223,85,0,electric_assist
2554,373273,204,0,electric_assist
...,...,...,...,...
87434,88184,119,0,electric_assist
87440,88194,119,0,electric_assist
87456,88248,119,0,electric_assist
87482,80279,85,0,electric_assist


In [102]:
bikes.query("computer_id != 0 & propulsion_type == 'electric_assist'")

,id,bike_type_id,computer_id,propulsion_type
1,374044,204,7551171765,electric_assist
2,373778,204,7551140444,electric_assist
4,88359,143,7551052355,electric_assist
5,374297,204,7551171579,electric_assist
6,373960,204,7551169969,electric_assist
...,...,...,...,...
88431,372994,204,7551151439,electric_assist
88432,372649,204,7551152435,electric_assist
88433,372951,204,7551150930,electric_assist
88434,374346,204,7551171592,electric_assist


In [104]:
bikes.query("computer_id == 0")

,id,bike_type_id,computer_id,propulsion_type
3,34749,14,0,human
9,75518,174,0,human
10,77035,4,0,human
11,28548,174,0,human
13,29302,4,0,human
...,...,...,...,...
87490,88247,119,0,electric_assist
87492,88149,14,0,human
87511,88229,14,0,human
88365,59522,4,0,human


In [124]:
bikes_with_same_computer = bikes.groupby('computer_id').agg(count=('computer_id', 'count')).sort_values(by = 'count', ascending = False)
bikes_with_same_computer.query("count > 1")

,count
computer_id,
0,13136
22577,5
20517,4
7551061141,4
21225,4
...,...
22332,2
7551091779,2
7551091778,2


In [133]:
bikes.query("bike_type_id == 0")

,id,bike_type_id,computer_id,propulsion_type
1408,604450,0,7551109281,human
1835,604553,0,7551112574,human
5996,8576,0,0,human
6602,76352,0,0,human
6873,8411,0,0,human
...,...,...,...,...
42265,67859,0,7551026571,human
46277,84983,0,7551121328,human
46335,84745,0,7551121666,human
46555,84702,0,7551124040,human


In [134]:
bike_types.query("id == 0")

,id,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity
82,0,https://static.nextbike.net/app/biketypes/type...,Unknown,Bikes with no bike type,bicycle,1.0,human,NaN,NaN
